# Widefield Local Analysis Walkthrough

This notebook is a step-by-step guide for reproducing the local `wfield` analysis and figures used for the PS94/PS95 examples. It is intentionally command-oriented: most cells print PowerShell commands that can be copied into a terminal, and optional Python cells inspect outputs after they are generated.

Recommended environment:

```powershell
conda activate wfield
cd "C:\\Github\\Widefield_DAQ_recorder"
```

Large outputs should stay with the experiment data, not in git.

## 0. Configure A Session

Edit the paths below for a new recording. For PS95, the latest reviewed alignment was `v6`; for PS94, the latest reviewed alignment was `v4`.

In [ ]:
from pathlib import Path
import json

repo = Path(r"C:/Github/Widefield_DAQ_recorder")

# Example: PS95
animal_label = "PS95_v6"
session = Path(r"E:/labcams_data/20260601/PS95_20260601_153653")
raw_dat = session / "raw_widefield_data" / "pco_edge_run000_00000000_2_540_640_uint16.dat"
camlog = session / "pco_edge_run000_00000000.camlog"
landmarks = session / "dorsal_cortex_landmarks.json"
daq_h5 = Path(r"E:/DAQ_recorder_output/PS95_baseline_20260601_153627.h5")

motion_dir = session / "motion_corrected"
motion_corrected_movie = motion_dir / "motioncorrect_2_540_640_uint16.bin"
wfield_results = motion_dir / "wfield_local_results"
allen_dir = wfield_results / "allen_aligned_v6"
cue_out = motion_dir / "spout_trial_averages_allen_v6"
lick_out = motion_dir / "lick_aligned_v6"

paired_frame_fs = 31.23  # hemodynamic-corrected 415/470-pair rate used for PS94/PS95
functional_channel = 1  # 0=415 nm, 1=470 nm for this rig

for name, path in {
    "repo": repo,
    "session": session,
    "raw_dat": raw_dat,
    "camlog": camlog,
    "landmarks": landmarks,
    "daq_h5": daq_h5,
}.items():
    print(f"{name:10s}: {path}")


## 1. Motion Correction

Run this only once per raw movie unless you want to test a different motion-correction mode.

Common parameters:

- `--mode 2d`: lean rigid XY correction; default practical option.
- `--mode ecc`: richer/slower option; test carefully on a subset first.
- `--output-dir`: session-specific output folder.

In [ ]:
print(fr'''python .\wfield_local\run_wfield_motion.py `
  "{raw_dat}" `
  --output-dir "{motion_dir}" `
  --mode 2d''')


## 2. SVD And Hemodynamic Correction

This step creates `U.npy`, `SVT.npy`, `SVTcorr.npy`, `frames_average.npy`, and related local results.

Important parameters:

- `--functional-channel 1`: 470 nm is channel 1 for this rig.
- `--n-components 100`: used for PS94/PS95.
- Memory/chunk parameters can be adjusted inside the script if needed.

In [ ]:
print(fr'''python .\wfield_local\run_wfield_local.py `
  "{motion_corrected_movie}" `
  --output-dir "{wfield_results}" `
  --functional-channel {functional_channel} `
  --n-components 100''')


## 3. Allen Alignment

After drawing or revising `dorsal_cortex_landmarks.json`, write a versioned Allen output folder. Keep versions separate when comparing alignment attempts.

The transform direction is subtle: the stored transform maps reference points toward clicked image points, while image warping uses it as an inverse map. The diagnostic plotting scripts handle this.

In [ ]:
print(fr'''python .\wfield_local\apply_allen_transform.py `
  "{wfield_results}" `
  --landmarks "{landmarks}" `
  --output "{allen_dir}"''')


## 4. Cue-Aligned Spout-Position Maps

This uses DAQ cue timing and the most recent `spout_strobe` code before each cue.

Default windows here are 1 s pre-cue and 1 s post-cue.

In [ ]:
print(fr'''python .\wfield_local\plot_spout_trial_averages.py `
  --label {animal_label} `
  --daq-h5 "{daq_h5}" `
  --wfield-results "{wfield_results}" `
  --allen-dir "{allen_dir}" `
  --camlog "{camlog}" `
  --output "{cue_out}" `
  --frame-align pco `
  --pre-s 1.0 `
  --post-s 1.0 `
  --fs {paired_frame_fs}''')


## 5. Shared-Scale Cue Maps

The original cue plot uses separate scales for pre/post and post-minus-pre. This plot forces one scale across all panels so visual comparisons are safer.

In [ ]:
cue_maps = cue_out / f"{animal_label}_spout_positions_1s_pre_post_delta_maps.npz"
cue_summary = cue_out / f"{animal_label}_spout_positions_1s_pre_post_delta_summary.json"
print(fr'''python .\wfield_local\plot_spout_trial_averages_shared_scale.py `
  --label {animal_label} `
  --trial-maps "{cue_maps}" `
  --allen-dir "{allen_dir}" `
  --summary "{cue_summary}" `
  --output "{cue_out}"''')


## 6. Cue Pairwise Spout-Position Contrasts

This creates contrasts such as close_L - close_R, close_center - close_R, close_L - far_L, etc., using the cue post-minus-pre maps.

In [ ]:
print(fr'''python .\wfield_local\plot_spout_position_contrasts.py `
  --label {animal_label} `
  --trial-maps "{cue_maps}" `
  --allen-dir "{allen_dir}" `
  --output "{cue_out}"''')


## 7. Alignment Diagnostics

Use these when comparing landmark attempts. The before/after plot shows where clicked landmarks start and where they land after the image-warp direction. The ROI label map shows the Allen/wfield segmented cortical areas.

In [ ]:
alignment_before_after = motion_dir / "alignment_before_after"
alignment_overlays = motion_dir / "alignment_landmark_overlays"
print(fr'''python .\wfield_local\plot_alignment_before_after.py `
  --label {animal_label.split('_')[0]} `
  --json-dir "{session}" `
  --results "{wfield_results}" `
  --output "{alignment_before_after}" `
  --current-version {animal_label.split('_')[-1]}''')
print()
print(fr'''python .\wfield_local\plot_alignment_landmark_overlays.py `
  --label {animal_label.split('_')[0]} `
  --json-dir "{session}" `
  --results "{wfield_results}" `
  --output "{alignment_overlays}" `
  --current-version {animal_label.split('_')[-1]}''')


In [ ]:
print(fr'''python .\wfield_local\plot_allen_reference_landmarks.py `
  --landmarks "{landmarks}" `
  --output "{session.parent / 'allen_wfield_reference_landmark_targets.png'}"''')
print()
print(fr'''python .\wfield_local\plot_allen_roi_labels.py `
  --allen-dir "{allen_dir}" `
  --output "{session.parent / (animal_label + '_allen_roi_labels.png')}" `
  --title "Allen/wfield ROI labels from {animal_label}"''')


## 8. Post-Lick Maps By Spout Position

Lick detection uses the imported double-threshold + lockout detector:

- onset: lick analog voltage drops below `--lick-thresh-upper-v`
- offset: voltage rises above `--lick-thresh-lower-v`
- offset lockout: drops noisy recontacts shortly after offset
- refractory: optional coarser event spacing for imaging averages

No pre-lick baseline is used in this figure.

In [ ]:
print(fr'''python .\wfield_local\plot_lick_aligned_averages.py `
  --label {animal_label} `
  --daq-h5 "{daq_h5}" `
  --wfield-results "{wfield_results}" `
  --allen-dir "{allen_dir}" `
  --camlog "{camlog}" `
  --output "{lick_out}" `
  --frame-align pco `
  --lick-thresh-upper-v 2.5 `
  --lick-thresh-lower-v 1.0 `
  --lockout-s 0.001 0.020 `
  --refractory-s 0.10 `
  --post-s 0.150 `
  --fs {paired_frame_fs}''')


## 9. Post-Lick Pairwise Spout-Position Contrasts

In [ ]:
lick_maps = lick_out / f"{animal_label}_lick_aligned_150ms_post_by_spout_maps.npz"
print(fr'''python .\wfield_local\plot_lick_position_contrasts.py `
  --label {animal_label} `
  --lick-maps "{lick_maps}" `
  --allen-dir "{allen_dir}" `
  --output "{lick_out}"''')


## 10. Cue vs Lick Comparison

Columns are cue post map, lick post map, and lick minus cue. Note that the default windows are different (`1 s post-cue` vs `150 ms post-lick`), so interpret the third column descriptively.

In [ ]:
lick_summary = lick_out / f"{animal_label}_lick_aligned_150ms_post_by_spout_summary.json"
print(fr'''python .\wfield_local\plot_lick_vs_cue_spout_maps.py `
  --label {animal_label} `
  --cue-maps "{cue_maps}" `
  --lick-maps "{lick_maps}" `
  --allen-dir "{allen_dir}" `
  --cue-summary "{cue_summary}" `
  --lick-summary "{lick_summary}" `
  --output "{lick_out}"''')


## 11. Build The Comparison PowerPoint

The PPT builder is currently configured for the PS94/PS95 June 1 example folders. For a new animal/session set, edit `wfield_local/build_alignment_comparison_ppt.ps1` or copy it to a new project-specific builder.

In [ ]:
print(fr'''powershell -NoProfile -ExecutionPolicy Bypass -File `
  "{repo / 'wfield_local' / 'build_alignment_comparison_ppt.ps1'}" `
  -OutputPath "{session.parent / 'alignment_comparison_PS94_PS95_with_allen_roi_labels.pptx'}"''')


## 12. Optional: Sync-Pulse DAQ ↔ Camera Alignment

`wfield_local/frame_sync.py` ports the stroke/orofacial sync-pulse algorithm. Use this when you have a camera-side sync column and a DAQ sync channel and want a pulse-pattern-derived mapping rather than wall-clock timestamp matching.

In [ ]:
from wfield_local.frame_sync import make_alignment_template

print('''# Example API; fill in real arrays/dataframe before running:
template = make_alignment_template(
    ws_signals={"Sync_signal": sync_trace, "sample_rate_input": 5000.0},
    cam_csv=cam_timestamp_dataframe,
    params={
        "csv_idx_sync": 0,
        "csv_idx_time": 1,
        "key_sync": "Sync_signal",
        "fps_cam": 62.46,
        "window": 20,
        "p": 0.1,
        "min_matched_edges": 5,
        "edge_threshold": 0.5,
    },
)
np.savez_compressed("alignment_template.npz", **template)
''')


## 13. Inspect Summaries And View Outputs

In [ ]:
for summary in [cue_summary, lick_summary, lick_out / f"{animal_label}_cue_vs_lick_spout_position_maps_summary.json"]:
    print('\n---', summary)
    if summary.exists():
        payload = json.loads(summary.read_text())
        for key in ['valid_cues_with_windows', 'valid_licks_with_windows', 'counts_by_position', 'display_limit', 'shared_display_limit']:
            if key in payload:
                print(key, payload[key])
    else:
        print('not found')
